In [ ]:
import sys, os
# Make the repo root importable (functions.py) and locate docs/, whether this
# notebook is launched from the repo root or from the otherModels/ subfolder.
_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'otherModels' else os.getcwd()
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import functions as fs
# GradientBoostingRegressor
from sklearn import ensemble
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold,cross_validate
from sklearn.metrics import make_scorer
import seaborn as sns

plt.close('all')
##################attempt number################################
# import datafile
file_path = os.path.join(_ROOT, 'docs', 'Data_base.xlsx')
# Read the Excel file into a pandas DataFrame
df = pd.read_excel(file_path, index_col=0)

In [ ]:
# Element columns = everything except the temperature feature (invT) and target (kp)
element_names = [c for c in df.columns if c not in ('invT', 'kp')]
print('Elemental features (%d):' % len(element_names), element_names)

In [ ]:
# Composition (varying elements) + test temperature (invT) -- no feature-engineering log,
# no external HEA descriptors. Consistent with GBR.ipynb / ModelSelection.ipynb.
composition_features = [e for e in element_names if df[e].nunique() > 1] + ['invT']
print('Features used for the model:', composition_features)

In [ ]:
#################################################################
trainset,testset = fs.data_split(df,element_names,0.2)

comp_major_low = -0.1
comp_major_high = 100.3
comp_major_inter = 10
comp_minor_low = -0.1
comp_minor_high = 50.3
comp_minor_inter = 0.1
T_low = 10
T_high = 2510
T_inter = 50
size = 8

Sampled_trainset = fs.data_sampling(trainset,comp_major_low,comp_major_high,comp_major_inter,
                                 comp_minor_low,comp_minor_high,comp_minor_inter,
                                 T_low,T_high,T_inter,size,element_names,random_state=42)
#################################################################

In [ ]:
# #################################################################
selected_columns_sampled_trainset = Sampled_trainset.loc[:, Sampled_trainset.columns.intersection(composition_features)]
selected_columns_sampled_trainset['kp'] = Sampled_trainset['kp']

selected_columns_testset = testset.loc[:, testset.columns.intersection(composition_features)]
selected_columns_testset['kp'] = testset['kp']
####################################
X_train = selected_columns_sampled_trainset.drop(columns=['kp'])  # Exclude the target column trainset
y_train = np.log10(selected_columns_sampled_trainset['kp'])#trainset
X_test = selected_columns_testset.drop(columns=['kp'])  # Exclude the target column
y_test = np.log10(selected_columns_testset['kp'])

In [ ]:
###################################################################################################################
from sklearn.svm import SVR
###################################################################################
params = {"kernel":"rbf","gamma":1e-2 ,"C":1, "epsilon":0.2,"tol":1e-4}#5e-3
# 'poly', 'rbf', 'precomputed', 'sigmoid', 'linear',"degree":10
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
reg = make_pipeline(StandardScaler(), SVR(**params))  # StandardScaler: SVR is scale-sensitive
###################################################################################################################
# Define the scoring metrics
scoring = {
    'MAE': make_scorer(mean_absolute_error),
    'MSE': make_scorer(mean_squared_error),
    'R2': make_scorer(r2_score)
}
# Perform five-fold cross-validation on the training set
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

cv_results = cross_validate(reg, X_train, y_train, cv=kfold, scoring=scoring)

# Extract and print the cross-validation results
print("Train CV MAE Score:", cv_results['test_MAE'].mean(),'±',cv_results['test_MAE'].std())
print("Train CV MSE Score:", cv_results['test_MSE'].mean(),'±',cv_results['test_MAE'].std())
print("Train CV R2 Score:", cv_results['test_R2'].mean(),'±',cv_results['test_MAE'].std())
#################################################################################################################
# Train the model using the training set
reg.fit(X_train, y_train)

# Make predictions on the testing data
y_test_pred = reg.predict(X_test)
# Calculate Mean Absolute Error (MAE)
mae = mean_absolute_error(y_test, y_test_pred)

# Calculate Mean Squared Error (MSE)
mse = mean_squared_error(y_test, y_test_pred)

# Calculate R-squared (R2)
r2 = r2_score(y_test, y_test_pred)

# Print the scores
print("Test Score Mean Absolute Error (MAE):", mae)
print("Test Score Mean Squared Error (MSE):", mse)
print("Test Score R-squared (R2):", r2)

# test_score = np.zeros((params["n_estimators"],), dtype=np.float64)
# for i, Y_pred in enumerate(reg.staged_predict(X_test)):
#     test_score[i] = mean_squared_error(y_test, Y_pred)

In [ ]:
#################################################################
y_train_pred = reg.predict(X_train)
y_test_pred = reg.predict(X_test)

plt.figure(figsize=(7,7))#
linex = [-16,-2]
liney = [-16,-2]
sns.set(style="darkgrid")
plt.scatter(x=y_train,y=y_train_pred,label="Training Set")#,c='b'
plt.scatter(x=y_test,y=y_test_pred,label="Test Set")#,marker='o',edgecolors='r',c='none'
plt.plot(linex,liney,linestyle='--',color='black')#
plt.ylim(-16,-2)
plt.xlim(-16,-2)
plt.ylabel("Prediction Oxidation Rate Constant log[$k_p]$ $(g^2cm^{-4}s^{-1})$")
plt.xlabel("True Oxidation Rate Constant log[$k_p]$ $(g^2cm^{-4}s^{-1})$")
plt.legend()
### Set a high DPI for better resolution (e.g., 300)
# # Save the plot to a file with high resolution
# plt.savefig('SVRperformance.png', bbox_inches='tight')
#################################################################

In [ ]:
# #################################################retain the model base on the entire database##############################
# Sampled_df = fs.data_sampling(df,comp_major_low,comp_major_high,comp_major_inter,
#                                  comp_minor_low,comp_minor_high,comp_minor_inter,
#                                  T_low,T_high,T_inter,size,element_names)
# # #################################################################
# selected_columns_sampled_df = Sampled_df.loc[:, Sampled_df.columns.intersection(composition_features)]
# selected_columns_sampled_df['kp'] = Sampled_df['kp']
# ####################################
# X_train_full = selected_columns_sampled_df.drop(columns=['kp'])  # Exclude the target column trainset
# y_train_full = np.log10(selected_columns_sampled_df['kp'])#trainset
# # Train the model using the training set
# reg.fit(X_train_full, y_train_full)

In [ ]:
#################SAVE-LOAD using pickle#####################
import joblib
# Save with the algorithm name in the filename so different models don't overwrite each other
MODEL_NAME = 'SVR'
out_path = f'{MODEL_NAME}.pkl'
joblib.dump(reg, out_path)
print('Saved', MODEL_NAME, 'model to', out_path)